# Inspeção de laranjas com YOLO: treino no Google Colab

Use este notebook se o seu computador não tiver GPU ou se o PyTorch não rodar nele.
O código é o mesmo do projeto: aqui só chamamos a CLI `citrus-yolo`.

**Antes de começar**
1. Em *Ambiente de execução > Alterar o tipo de ambiente*, escolha **GPU (T4)**.
2. Suba a pasta do projeto para o Google Drive, **sem** a `.venv`, em `MeuDrive/citrus-yolo-inspection`,
   já com as fotos em `dataset/bruto` e as inéditas em `dataset/ineditas`.

Como tudo roda dentro do Drive, os resultados (best.pt, métricas, relatório, ZIP) já ficam salvos lá.

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/citrus-yolo-inspection

## 1. Instalação e verificação do YOLO

In [ ]:
!pip install -q -e ".[dev]"
!yolo checks

## 2. Dataset: divisão 80/20, data.yaml e checagens

In [ ]:
!citrus-yolo preparar

## 3. Treino (30 épocas, yolov8n.pt, imgsz 640)
Na GPU do Colab dá para usar um batch maior. **Tire um print desta célula durante o treino** e salve em `resultados/prints/`.

In [ ]:
!citrus-yolo treinar --dispositivo 0 --batch 16

## 4. Validação: Precision, Recall, mAP50

In [ ]:
!citrus-yolo validar --dispositivo 0

## 5. Detecção nas imagens inéditas

In [ ]:
!citrus-yolo detectar --dispositivo 0

In [ ]:
from IPython.display import Image, display
display(Image('resultados/treino/results.png', width=900))
display(Image('resultados/deteccoes/mosaico_deteccoes.jpg', width=900))

## 6. Relatório e ZIP
Antes, edite `configs/relatorio.yaml` com as suas palavras (dá para editar pelo próprio Drive).

In [ ]:
!citrus-yolo relatorio
!citrus-yolo entregar

### Comandos equivalentes na CLI do Ultralytics (os do enunciado)
```
yolo detect train model=yolov8n.pt data="dataset/data.yaml" epochs=30 imgsz=640 batch=4 project="$(pwd)/runs/detect" name=train
yolo detect val model="runs/detect/train/weights/best.pt" data="dataset/data.yaml" project="$(pwd)/runs/detect" name=val
yolo detect predict model="runs/detect/train/weights/best.pt" source="dataset/ineditas" conf=0.25 save=True project="$(pwd)/runs/detect" name=predict
```